# Create Nobel Awards (PRIZE PATTERN TEMPLATE)

Creates Nobel Prize laureate rows for the four science categories
(Physics, Chemistry, Economic Sciences, Physiology or Medicine) from
the official Nobel API at api.nobelprize.org/2.1.

**This is a PRIZE pattern, not a grant pattern.** Schema differences vs.
the standard award flow:
- One row per (prize × laureate). Shared prizes produce N rows, with
  `portion` indicating each laureate's share.
- `amount` = nominal SEK in the year of award. The Nobel API also
  exposes `prizeAmountAdjusted` (inflation-adjusted to ~2023 SEK); we
  keep that as a separate column for analysis.
- `currency` = "SEK" (always — Nobel awards are paid in Swedish kronor).
- `funder` is the *awarding body* (which differs by category), NOT a
  hypothetical "Nobel Foundation". Nobel Foundation is not a funder
  entity in OpenAlex.

**Funder mapping (category → OpenAlex funder_id):**
- Physics, Chemistry, Economic Sciences → Royal Swedish Academy of
  Sciences (`F4320320936`)
- Physiology or Medicine → Karolinska Institutet (`F4320322315`)

Out of scope: Literature (Swedish Academy) and Peace (Norwegian Nobel
Committee) are excluded — neither awarding body is registered as a
funder in OpenAlex, and the awards aren't research-funding in the
OpenAlex sense.

**Prerequisites:**
- Run `scripts/local/nobel_to_s3.py` to download laureate data and
  upload the parquet first.

**Data source:** api.nobelprize.org/2.1
**S3 location:** `s3a://openalex-ingest/awards/nobel/nobel_prizes.parquet`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nobel_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nobel/nobel_prizes.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.nobel_raw;

## Step 1.5: Inspect raw + verify amount/currency

For prize patterns, amount may be NULL for non-monetary prizes (Fields
Medal, etc.). Nobel does have a per-year amount, so we expect 100%
coverage.

In [ ]:
%sql
DESCRIBE openalex.awards.nobel_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.nobel_raw LIMIT 5;

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(prize_amount_sek) AS has_nominal_amt,
  COUNT(prize_amount_adjusted_sek) AS has_adjusted_amt,
  ROUND(MIN(prize_amount_sek), 0) AS min_nominal,
  ROUND(MAX(prize_amount_sek), 0) AS max_nominal,
  collect_set(category_code) AS categories
FROM openalex.awards.nobel_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nobel_awards
USING delta
AS
WITH cat_funder_map AS (
    SELECT * FROM (VALUES
        ('phy', 4320320936),
        ('che', 4320320936),
        ('eco', 4320320936),
        ('med', 4320322315)
    ) AS t(category_code, funder_id)
),
funders_resolved AS (
    SELECT m.category_code, f.funder_id, f.display_name, f.ror_id, f.doi
    FROM cat_funder_map m
    JOIN openalex.common.funder f USING (funder_id)
)
SELECT
    -- Unique ID: hash of awarding-body funder_id + nobel category + year + laureate id.
    -- This makes shared-prize rows distinct (different laureate_id) while shared by year+cat.
    abs(xxhash64(CONCAT(
        f.funder_id, ':nobel:', n.category_code, ':', n.award_year, ':', n.laureate_id
    ))) % 9000000000 AS id,
    -- "{Category} {Year} — {Laureate name}"
    CONCAT(
        n.category_full_en, ' ', n.award_year,
        ' — ', n.laureate_full_name
    ) AS display_name,
    n.motivation_en AS description,
    f.funder_id,
    -- Award ID is the Nobel API's stable laureate_id + year + category code
    CONCAT(n.category_code, '-', n.award_year, '-', n.laureate_id) AS funder_award_id,
    -- Apportioned amount: full SEK amount × portion (parsed from "1/2" etc.)
    CASE
        WHEN n.portion = '1'   THEN TRY_CAST(n.prize_amount_sek AS DOUBLE)
        WHEN n.portion = '1/2' THEN TRY_CAST(n.prize_amount_sek AS DOUBLE) * 0.5
        WHEN n.portion = '1/3' THEN TRY_CAST(n.prize_amount_sek AS DOUBLE) / 3.0
        WHEN n.portion = '1/4' THEN TRY_CAST(n.prize_amount_sek AS DOUBLE) * 0.25
        ELSE TRY_CAST(n.prize_amount_sek AS DOUBLE)
    END AS amount,
    'SEK' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.category_full_en AS funder_scheme,
    'nobelprize_api' AS provenance,
    -- Date awarded; fallback to Dec 10 (traditional Nobel ceremony date)
    COALESCE(
        TRY_TO_DATE(n.date_awarded, 'yyyy-MM-dd'),
        TRY_TO_DATE(CONCAT(n.award_year, '-12-10'), 'yyyy-MM-dd')
    ) AS start_date,
    -- Prize is a one-time award, end_date = start_date
    COALESCE(
        TRY_TO_DATE(n.date_awarded, 'yyyy-MM-dd'),
        TRY_TO_DATE(CONCAT(n.award_year, '-12-10'), 'yyyy-MM-dd')
    ) AS end_date,
    TRY_CAST(n.award_year AS INT) AS start_year,
    TRY_CAST(n.award_year AS INT) AS end_year,
    -- Laureate IS the lead investigator. Affiliation if known, else NULL.
    struct(
        n.laureate_given_name AS given_name,
        n.laureate_family_name AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.affiliation_name AS name,
            n.affiliation_country AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CONCAT('https://www.nobelprize.org/prizes/', n.category_code, '/', n.award_year, '/summary/') AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(
               f.funder_id, ':nobel:', n.category_code, ':', n.award_year, ':', n.laureate_id
           ))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.nobel_raw n
JOIN funders_resolved f ON n.category_code = f.category_code
WHERE n.laureate_id IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 42

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nobelprize_api' AND priority = 42;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    42 AS priority
FROM openalex.awards.nobel_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_nobel_award_rows FROM openalex.awards.nobel_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
-- Note: prizes ALWAYS have amount + currency in the Nobel case; for
-- prizes that don't (e.g. Fields Medal — non-monetary), amount can be NULL
-- and the check should be relaxed to "currency-or-explicit-non-monetary".
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt
FROM openalex.awards.nobel_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS category, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name AS affiliation
FROM openalex.awards.nobel_awards
ORDER BY start_year DESC LIMIT 15;

In [ ]:
%sql
-- Distribution by category and awarding body
SELECT funder.display_name, funder_scheme AS category, COUNT(*) AS cnt
FROM openalex.awards.nobel_awards
GROUP BY funder.display_name, funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Year coverage
SELECT MIN(start_year) AS first_year, MAX(start_year) AS last_year, COUNT(DISTINCT start_year) AS distinct_years
FROM openalex.awards.nobel_awards;